# preprocessing

In [ ]:
!pip install -q  pandas scikit-learn tqdm tensorflow
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 

import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Concatenate, Dropout, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

#  float32，
tf.keras.backend.set_floatx('float32')

# ———— Kaggle  ————
RAW_DATA = 'raw_data_final_40501.json'
OUT_CV   = 'textcnn_10.txt'
OUT_LOGO = 'textcnn_logo.txt'

# ———— TextCNN  ————
MAX_WORDS = 20000
MAX_LEN   = 100
EMB_DIM   = 300
FILTER_SIZES = (1, 2, 3, 4, 5)
NUM_FILTERS  = 200
DROPOUT_RATE = 0.5
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
EPOCHS = 50

# ————  ————
with open(RAW_DATA, 'r', encoding='utf-8') as f:
    records = json.load(f)
df = pd.DataFrame(records)
texts    = df['f_comment'].astype(str).tolist()
labels   = df['is_self_fixed'].astype(int).values
projects = df['project_name'].astype(str).tolist()

# ————  ————
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
data = pad_sequences(sequences, maxlen=MAX_LEN)

# ———— TextCNN  ————
def build_textcnn():
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(input_dim=MAX_WORDS, output_dim=EMB_DIM)(inp)
    convs = []
    for sz in FILTER_SIZES:
        c = Conv1D(filters=NUM_FILTERS, kernel_size=sz, activation='relu')(x)
        p = GlobalMaxPooling1D()(c)
        convs.append(p)
    x = Concatenate()(convs)
    x = Dropout(DROPOUT_RATE)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy')
    return model

print("preprocessing Done...")

/root/miniconda3/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violat

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
preprocessing Done...


# 10fold

In [ ]:
# from tensorflow.keras.callbacks import EarlyStopping

# # ———— 10  ————
# kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# cv_metrics = {'auc':[], 'acc':[], 'prec':[], 'recall':[], 'f1':[]}

# for fold, (tr_idx, te_idx) in enumerate(kf.split(data, labels), 1):
#     model = build_textcnn()
    
#     # ✳️ （）
#     early_stop = EarlyStopping(
#         monitor='loss',
#         patience=3,                # 3
#         restore_best_weights=True, # 
#         verbose=1
#     )
    
#     # ✳️  callbacks
#     model.fit(
#         data[tr_idx], labels[tr_idx],
#         epochs=EPOCHS,
#         batch_size=BATCH_SIZE,
#         verbose=1,
#         callbacks=[early_stop]
#     )
    
#     y_proba = model.predict(data[te_idx], batch_size=BATCH_SIZE).ravel()
#     y_pred  = (y_proba >= 0.5).astype(int)
#     cv_metrics['auc'].append(roc_auc_score(labels[te_idx], y_proba))
#     cv_metrics['acc'].append(accuracy_score(labels[te_idx], y_pred))
#     cv_metrics['prec'].append(precision_score(labels[te_idx], y_pred, zero_division=0))
#     cv_metrics['recall'].append(recall_score(labels[te_idx], y_pred, zero_division=0))
#     cv_metrics['f1'].append(f1_score(labels[te_idx], y_pred, zero_division=0))
#     print(f"Fold{fold}: AUC={cv_metrics['auc'][-1]:.4f}")

# # ————  ————
# with open(OUT_CV, 'w', encoding='utf-8') as f:
#     for i in range(10):
#         f.write(
#             f"Fold{i+1}: AUC={cv_metrics['auc'][i]:.4f}, "
#             f"Acc={cv_metrics['acc'][i]:.4f}, "
#             f"Prec={cv_metrics['prec'][i]:.4f}, "
#             f"Rec={cv_metrics['recall'][i]:.4f}, "
#             f"F1={cv_metrics['f1'][i]:.4f}\n"
#         )
#     avg = {m: np.mean(v) for m, v in cv_metrics.items()}
#     f.write(
#         f": AUC={avg['auc']:.4f}, Acc={avg['acc']:.4f}, "
#         f"Prec={avg['prec']:.4f}, Rec={avg['recall']:.4f}, F1={avg['f1']:.4f}\n"
#     )
# print("10CV", OUT_CV)

# lopo

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# ————  (LOPO)  ————
logo_metrics = {'project': [], 'auc': [], 'acc': [], 'prec': [], 'recall': [], 'f1': []}

# 
unique_projects = np.unique(projects)

selected_projects = unique_projects[38:50]  # 3950

print(f" {len(selected_projects)} LOPO")

for proj in selected_projects:
    idx_tr = np.where(np.array(projects) != proj)[0]
    idx_te = np.where(np.array(projects) == proj)[0]
    X_tr, y_tr = data[idx_tr], labels[idx_tr]
    X_te, y_te = data[idx_te], labels[idx_te]
    model = build_textcnn()

    # ✳️ （）
    early_stop = EarlyStopping(
        monitor='loss',
        patience=3,                # 3
        restore_best_weights=True, # 
        verbose=1
    )

    # ✳️ fitcallbacks
    model.fit(
        X_tr, y_tr,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
        callbacks=[early_stop]
    )

    y_proba = model.predict(X_te, batch_size=BATCH_SIZE).ravel()
    y_pred = (y_proba >= 0.5).astype(int)

    logo_metrics['project'].append(proj)
    logo_metrics['auc'].append(roc_auc_score(y_te, y_proba))
    logo_metrics['acc'].append(accuracy_score(y_te, y_pred))
    logo_metrics['prec'].append(precision_score(y_te, y_pred, zero_division=0))
    logo_metrics['recall'].append(recall_score(y_te, y_pred, zero_division=0))
    logo_metrics['f1'].append(f1_score(y_te, y_pred, zero_division=0))
    print(f"Proj:{proj} AUC={logo_metrics['auc'][-1]:.4f}")
    safe_proj = proj.replace('/', '_').replace('\\', '_')  #  Windows  Linux
    with open(f'{safe_proj}.txt', 'w', encoding='utf-8') as f:
        f.write(
            f"Proj{proj}: "
            f"AUC:{roc_auc_score(y_te, y_proba):.4f}, "
            f"Acc:{accuracy_score(y_te, y_pred):.4f}, "
            f"Prec:{precision_score(y_te, y_pred, zero_division=0):.4f}, "
            f"Rec:{recall_score(y_te, y_pred, zero_division=0):.4f}, "
            f"F1:{f1_score(y_te, y_pred, zero_division=0):.4f}\n"
        )

# 
with open(OUT_LOGO, 'w', encoding='utf-8') as f:
    for i, proj in enumerate(logo_metrics['project']):
        f.write(
            f"Proj{proj}: "
            f"AUC:{logo_metrics['auc'][i]:.4f}, "
            f"Acc:{logo_metrics['acc'][i]:.4f}, "
            f"Prec:{logo_metrics['prec'][i]:.4f}, "
            f"Rec:{logo_metrics['recall'][i]:.4f}, "
            f"F1:{logo_metrics['f1'][i]:.4f}\n"
        )
    avg_logo = {m: np.mean(v) for m, v in logo_metrics.items() if m != 'project'}
    f.write(
        f"(50): "
        f"AUC:{avg_logo['auc']:.4f}, "
        f"Acc:{avg_logo['acc']:.4f}, "
        f"Prec:{avg_logo['prec']:.4f}, "
        f"Rec:{avg_logo['recall']:.4f}, "
        f"F1:{avg_logo['f1']:.4f}\n"
    )

print("", OUT_LOGO)

共选择 12 个项目用于LOPO验证


I0000 00:00:1760942528.656867    1238 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22163 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:1c:00.0, compute capability: 8.9


Epoch 1/50
  32/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.6920

I0000 00:00:1760942536.872292    1550 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1251/1251 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - loss: 0.6792
Epoch 2/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.6725
Epoch 3/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.6664
Epoch 4/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.6594
Epoch 5/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.6511
Epoch 6/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.6408
Epoch 7/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.6281
Epoch 8/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.6136
Epoch 9/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.5963
Epoch 10/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.5795
Epoch 11/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.5628
Epoch 12/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.5456
Epoch 13/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.5277
Epoch 14/50
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.5114
Epoch 15/50
1251/1251 ━━━━━━━━━━━━━━━━━

In [ ]:
import os, re, glob
import numpy as np

# ————  ————
metric_line_re = re.compile(
    r"^\s*Proj[:\s]*(?P<proj>[^:]+?):?\s*"  # Proj 
    r"AUC[:=](?P<auc>[0-9]*\.?[0-9]+).*?"     # AUC
    r"Acc[:=](?P<acc>[0-9]*\.?[0-9]+).*?"     # Acc
    r"Prec[:=](?P<prec>[0-9]*\.?[0-9]+).*?"   # Prec
    r"Rec[:=](?P<rec>[0-9]*\.?[0-9]+).*?"     # Rec
    r"F1[:=](?P<f1>[0-9]*\.?[0-9]+)\s*$",    # F1
    flags=re.IGNORECASE
)

def parse_metrics_from_text(text: str):
    results = []
    for line in text.splitlines():
        m = metric_line_re.search(line)
        if m:
            proj = m.group('proj').strip()
            results.append(
                (
                    proj,
                    {
                        'auc': float(m.group('auc')),
                        'acc': float(m.group('acc')),
                        'prec': float(m.group('prec')),
                        'recall': float(m.group('rec')),
                        'f1': float(m.group('f1')),
                    },
                )
            )
    return results

# ———— ，（） ————
# ：dl_results/*.txt > textcnn_lopo_55.txt > textcnn_logo.txt
sources = []
if os.path.exists('textcnn_logo.txt'):
    sources.append(('logo', 'textcnn_logo.txt'))
if os.path.exists('textcnn_lopo_55.txt'):
    sources.append(('lopo55', 'textcnn_lopo_55.txt'))
if os.path.isdir('dl_results'):
    for p in sorted(glob.glob(os.path.join('dl_results', '*.txt'))):
        sources.append(('dl', p))

project_to_metrics = {}
for tag, path in sources:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
        for proj, metrics in parse_metrics_from_text(text):
            # ：（dl）
            project_to_metrics[proj] = metrics
    except Exception as e:
        print(f" {path}: {e}")

num_projects = len(project_to_metrics)
print(f" {num_projects} ")

# ———— （） ————
if num_projects == 0:
    raise RuntimeError('。')

metrics_array = {
    'auc': np.array([m['auc'] for m in project_to_metrics.values()], dtype=np.float64),
    'acc': np.array([m['acc'] for m in project_to_metrics.values()], dtype=np.float64),
    'prec': np.array([m['prec'] for m in project_to_metrics.values()], dtype=np.float64),
    'recall': np.array([m['recall'] for m in project_to_metrics.values()], dtype=np.float64),
    'f1': np.array([m['f1'] for m in project_to_metrics.values()], dtype=np.float64),
}

avg = {k: float(np.mean(v)) for k, v in metrics_array.items()}

# ————  textcnn_lopo.txt（，） ————
with open('textcnn_lopo.txt', 'w', encoding='utf-8') as f:
    for proj in sorted(project_to_metrics.keys()):
        m = project_to_metrics[proj]
        f.write(
            f"Proj{proj}: "
            f"AUC:{m['auc']:.4f}, "
            f"Acc:{m['acc']:.4f}, "
            f"Prec:{m['prec']:.4f}, "
            f"Rec:{m['recall']:.4f}, "
            f"F1:{m['f1']:.4f}\n"
        )
    f.write(
        f"(={num_projects}): "
        f"AUC:{avg['auc']:.4f}, "
        f"Acc:{avg['acc']:.4f}, "
        f"Prec:{avg['prec']:.4f}, "
        f"Rec:{avg['recall']:.4f}, "
        f"F1:{avg['f1']:.4f}\n"
    )

print(' textcnn_lopo.txt')

汇总到 105 个唯一项目
项目逐行与平均结果已保存至 textcnn_lopo.txt
